In [1]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import hashlib
import platform
import importlib.metadata as package_metadata
import pandas as pd
import torch
from huggingface_hub import HfApi

BASELINE_ROOT = Path(
    "/kaggle/input/datasets/muusamunufo/base-1-trained/"
    "HealthVer_Experiments"
)
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
EXPERIMENT_ROOT = Path("/kaggle/working/HealthVer_TinyLlama_Experiments")

expected = {
    "train": ("train", 10590,
        "57668949f79d0f46e66d74df4b0bc9812cad4d5ad7c5dd03dfba008a608b3f2a"),
    "validation": ("dev", 1917,
        "6f8ea31cbcf2ea5a2d3c82d65587d66e96de4f70e72642bbf7f61fec6ef8a93a"),
}

raw_frames = {}
for split, (suffix, count, expected_hash) in expected.items():
    matches = list(
        (BASELINE_ROOT / "original_data").glob(f"*healthver_{suffix}.csv")
    )
    assert len(matches) == 1, f"{split}: expected exactly one CSV"
    path = matches[0]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected_hash
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert len(frame) == count
    assert {"id", "claim", "evidence", "label"} <= set(frame.columns)
    raw_frames[split] = frame
    print(f"{split}: {count:,} rows; original CSV hash verified")

print("\nPython:", platform.python_version())
for name in ["torch", "transformers", "peft", "accelerate",
             "datasets", "scikit-learn", "torchao"]:
    try:
        print(f"{name}: {package_metadata.version(name)}")
    except package_metadata.PackageNotFoundError:
        print(f"{name}: NOT INSTALLED")

assert torch.cuda.is_available(), "Enable the Kaggle GPU accelerator."
for i in range(torch.cuda.device_count()):
    gpu = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {gpu.name}, {gpu.total_memory / 1024**3:.2f} GiB")

MODEL_REVISION = HfApi().model_info(MODEL_ID).sha
assert MODEL_REVISION
print("\nNext model:", MODEL_ID)
print("Resolved model revision:", MODEL_REVISION)
print("Planned output:", EXPERIMENT_ROOT)
print("\nCheck complete. No weights loaded or experiment files written.")
print("No packages changed. Test contents not accessed.")

train: 10,590 rows; original CSV hash verified
validation: 1,917 rows; original CSV hash verified

Python: 3.13.15
torch: 2.11.0+cu128
transformers: 5.16.1
peft: 0.20.0
accelerate: 1.14.0
datasets: 4.8.5
scikit-learn: 1.6.1
torchao: NOT INSTALLED
GPU 0: Tesla T4, 14.56 GiB
GPU 1: Tesla T4, 14.56 GiB

Next model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Resolved model revision: fe8a4ea1ffedaf415f4da2f062534de366a451e6
Planned output: /kaggle/working/HealthVer_TinyLlama_Experiments

Check complete. No weights loaded or experiment files written.
No packages changed. Test contents not accessed.


In [3]:
import json
import pandas as pd
from transformers import AutoTokenizer
from tqdm.auto import tqdm

assert MODEL_REVISION == "fe8a4ea1ffedaf415f4da2f062534de366a451e6"

baseline_input = json.loads(
    (BASELINE_ROOT / "input_config.json").read_text()
)
SYSTEM_PROMPT = baseline_input["system_prompt"]
LABELS = baseline_input["labels"]
assert LABELS == ["Supports", "Refutes", "Neutral"]

SEQUENCE_LIMIT = 384
MAX_NEW_TOKENS = 12

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    use_fast=True,
)
assert tokenizer.chat_template
assert tokenizer.eos_token_id is not None
assert tokenizer.pad_token_id is not None
tokenizer.padding_side = "right"

def make_messages(claim, evidence):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Claim: {claim}\n\nEvidence: {evidence}",
        },
    ]

def checked_encoding(messages, label):
    assert label in LABELS
    prompt_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=False,
    )
    completed_ids = tokenizer.apply_chat_template(
        messages + [{"role": "assistant", "content": label}],
        tokenize=True,
        add_generation_prompt=False,
        return_dict=False,
    )
    assert isinstance(prompt_ids, list)
    assert isinstance(completed_ids, list)
    assert completed_ids[:len(prompt_ids)] == prompt_ids, (
        "Assistant boundary mismatch; stop and share this output."
    )

    answer_ids = completed_ids[len(prompt_ids):]
    assert answer_ids.count(tokenizer.eos_token_id) == 1
    decoded = tokenizer.decode(
        answer_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    ).strip()
    assert decoded == label, (label, answer_ids, decoded)
    return prompt_ids, completed_ids, answer_ids

example = raw_frames["train"].iloc[0]
messages = make_messages(example["claim"], example["evidence"])

print("Assistant label encodings:")
for label in LABELS:
    _, _, answer_ids = checked_encoding(messages, label)
    print(label, answer_ids, repr(tokenizer.decode(
        answer_ids, skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )))

reports = []
for split in ["train", "validation"]:
    frame = raw_frames[split]
    complete_lengths, generation_lengths = [], []
    answer_lengths = set()

    for row in tqdm(
        frame.itertuples(index=False),
        total=len(frame),
        desc=f"Checking {split}",
    ):
        prompt_ids, completed_ids, answer_ids = checked_encoding(
            make_messages(row.claim, row.evidence), row.label
        )
        complete_lengths.append(len(completed_ids))
        generation_lengths.append(len(prompt_ids) + MAX_NEW_TOKENS)
        answer_lengths.add(len(answer_ids))

    reports.append({
        "split": split,
        "rows": len(frame),
        "max_complete_tokens": max(complete_lengths),
        "complete_rows_over_384": sum(
            n > SEQUENCE_LIMIT for n in complete_lengths
        ),
        "max_prompt_plus_12": max(generation_lengths),
        "supervised_token_counts": sorted(answer_lengths),
    })

length_report = pd.DataFrame(reports)
print("\n" + length_report.to_string(index=False))

# Tokenizer-specific limit; preserve all original text and rows.
SEQUENCE_LIMIT = 448

assert all(
    r["max_complete_tokens"] <= SEQUENCE_LIMIT
    for r in reports
), "A complete example exceeds the new limit."

assert all(
    r["max_prompt_plus_12"] <= SEQUENCE_LIMIT
    for r in reports
), "A prompt plus generation allowance exceeds the new limit."

print(f"\nVerified TinyLlama sequence limit: {SEQUENCE_LIMIT}")
print("The table above records exceedances of the original 384-token limit.")
print("All prompt boundaries and answer EOS checks passed.")
print("All training and validation rows retained without truncation.")
print("No model weights loaded or experiment files written.")
print("Test contents not accessed.")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Assistant label encodings:
Supports [14039, 29879, 2, 13] 'Supports</s>\n'
Refutes [5620, 2667, 2, 13] 'Refutes</s>\n'
Neutral [8139, 329, 1705, 2, 13] 'Neutral</s>\n'


Checking train:   0%|          | 0/10590 [00:00<?, ?it/s]

Checking validation:   0%|          | 0/1917 [00:00<?, ?it/s]


     split  rows  max_complete_tokens  complete_rows_over_384  max_prompt_plus_12 supervised_token_counts
     train 10590                  409                       9                 417                  [4, 5]
validation  1917                  402                       1                 410                  [4, 5]

Verified TinyLlama sequence limit: 448
The table above records exceedances of the original 384-token limit.
All prompt boundaries and answer EOS checks passed.
All training and validation rows retained without truncation.
No model weights loaded or experiment files written.
Test contents not accessed.


In [4]:
import json
import hashlib
import shutil
from datasets import Dataset, load_from_disk
from tqdm.auto import tqdm

assert SEQUENCE_LIMIT == 448
assert MODEL_REVISION == "fe8a4ea1ffedaf415f4da2f062534de366a451e6"
assert not EXPERIMENT_ROOT.exists(), (
    f"Output already exists; inspect before overwriting: {EXPERIMENT_ROOT}"
)

DATASET_REVISION = baseline_input["dataset_revision"]
assert DATASET_REVISION == "b20ac99ceed62f5264a319fa25a854df1668d85b"

def content_sha256(dataset):
    digest = hashlib.sha256()
    for row in dataset:
        digest.update(
            (json.dumps(row, sort_keys=True, separators=(",", ":"))
             + "\n").encode("utf-8")
        )
    return digest.hexdigest()

tokenized = {}
split_metadata = {}
source_paths = {}

# Prepare and validate everything before writing.
for split in ["train", "validation"]:
    frame = raw_frames[split]
    suffix, expected_rows, expected_hash = expected[split]
    matches = list(
        (BASELINE_ROOT / "original_data").glob(f"*healthver_{suffix}.csv")
    )
    assert len(matches) == 1
    source_paths[split] = matches[0]
    source_hash = hashlib.sha256(matches[0].read_bytes()).hexdigest()
    assert source_hash == expected_hash
    assert len(frame) == expected_rows

    records = []
    supervised_counts = set()

    for row in tqdm(
        frame.itertuples(index=False),
        total=len(frame),
        desc=f"Preparing {split}",
    ):
        prompt_ids, complete_ids, answer_ids = checked_encoding(
            make_messages(row.claim, row.evidence), row.label
        )
        assert len(complete_ids) <= SEQUENCE_LIMIT
        assert len(prompt_ids) + MAX_NEW_TOKENS <= SEQUENCE_LIMIT

        records.append({
            "id": str(row.id),
            "input_ids": complete_ids,
            "attention_mask": [1] * len(complete_ids),
            "labels": [-100] * len(prompt_ids) + answer_ids,
            "sequence_length": len(complete_ids),
        })
        supervised_counts.add(len(answer_ids))

    dataset = Dataset.from_list(records)
    assert list(dataset["id"]) == frame["id"].tolist()
    tokenized[split] = dataset
    split_metadata[split] = {
        "rows": len(dataset),
        "source_csv_sha256": source_hash,
        "dataset_fingerprint": dataset._fingerprint,
        "tokenized_content_sha256": content_sha256(dataset),
        "max_sequence_length": max(dataset["sequence_length"]),
        "supervised_token_counts": sorted(supervised_counts),
    }
    del records

EXPERIMENT_ROOT.mkdir(parents=True)
(EXPERIMENT_ROOT / "original_data").mkdir()

for split, dataset in tokenized.items():
    destination = EXPERIMENT_ROOT / f"tokenized_{split}"
    dataset.save_to_disk(str(destination))

    reloaded = load_from_disk(str(destination))
    assert content_sha256(reloaded) == (
        split_metadata[split]["tokenized_content_sha256"]
    )
    saved_state = json.loads((destination / "state.json").read_text())
    assert saved_state["_fingerprint"] == dataset._fingerprint
    # Loaded cache fingerprints need not equal persisted fingerprints.
    del reloaded

    shutil.copy2(
        source_paths[split],
        EXPERIMENT_ROOT / "original_data" / source_paths[split].name,
    )

tokenizer.save_pretrained(str(EXPERIMENT_ROOT / "tokenizer"))

input_config = {
    **baseline_input,
    "model": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "max_sequence_length": SEQUENCE_LIMIT,
    "truncation": False,
    "padding": "dynamic",
    "loss": "assistant_answer_only",
}

preprocessing = {
    "model": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset_revision": DATASET_REVISION,
    "sequence_limit": SEQUENCE_LIMIT,
    "sequence_limit_reason": (
        "Preserve complete HealthVer text with TinyLlama tokenization; "
        "train/validation maximum complete length 409, prompt plus 12: 417."
    ),
    "truncation": False,
    "rows_removed": 0,
    "splits": split_metadata,
}

for filename, payload in [
    ("input_config.json", input_config),
    ("preprocessing_manifest.json", preprocessing),
]:
    (EXPERIMENT_ROOT / filename).write_text(
        json.dumps(payload, indent=2), encoding="utf-8"
    )

shutil.copy2(
    BASELINE_ROOT / "dataset_metadata.json",
    EXPERIMENT_ROOT / "dataset_metadata.json",
)

full_train = tokenized["train"]
full_validation = tokenized["validation"]

print(json.dumps(split_metadata, indent=2))
print("Saved:", EXPERIMENT_ROOT)
print("Saved tokenized contents verified after reload.")
print("No model loaded or training performed. Test data not accessed.")

Preparing train:   0%|          | 0/10590 [00:00<?, ?it/s]

Preparing validation:   0%|          | 0/1917 [00:00<?, ?it/s]

Saving the dataset (0/1 shards):   0%|          | 0/10590 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1917 [00:00<?, ? examples/s]

{
  "train": {
    "rows": 10590,
    "source_csv_sha256": "57668949f79d0f46e66d74df4b0bc9812cad4d5ad7c5dd03dfba008a608b3f2a",
    "dataset_fingerprint": "f403508492ed587a",
    "tokenized_content_sha256": "0a5aa867b0af171e3f747dd70a9abedf0c857dc387edac59347616c6d1da4c48",
    "max_sequence_length": 409,
    "supervised_token_counts": [
      4,
      5
    ]
  },
  "validation": {
    "rows": 1917,
    "source_csv_sha256": "6f8ea31cbcf2ea5a2d3c82d65587d66e96de4f70e72642bbf7f61fec6ef8a93a",
    "dataset_fingerprint": "a0232c7cbcd4435d",
    "tokenized_content_sha256": "425b70a4266588967d2506881090f43a83bb1ead14ff905d6ccbc8ec02589e38",
    "max_sequence_length": 402,
    "supervised_token_counts": [
      4,
      5
    ]
  }
}
Saved: /kaggle/working/HealthVer_TinyLlama_Experiments
Saved tokenized contents verified after reload.
No model loaded or training performed. Test data not accessed.


In [5]:
import json
import hashlib
import torch
from transformers import AutoModelForCausalLM, set_seed
from peft import LoraConfig, get_peft_model

SEED = 42
DEVICE = torch.device("cuda:0")
INITIAL_ADAPTER_DIR = EXPERIMENT_ROOT / "initial_adapter_seed42"
INITIALIZATION_PATH = EXPERIMENT_ROOT / "initialization_manifest.json"

assert not INITIAL_ADAPTER_DIR.exists(), "Initial adapter already exists."
assert not INITIALIZATION_PATH.exists(), "Initialization manifest already exists."
assert MODEL_REVISION == "fe8a4ea1ffedaf415f4da2f062534de366a451e6"

set_seed(SEED)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=torch.float16,
    device_map={"": 0},
    attn_implementation="sdpa",
)
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

# Explicitly fix randomness immediately before adapter initialization.
set_seed(SEED)
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
    init_lora_weights=True,
    revision=MODEL_REVISION,
)
model = get_peft_model(model, lora_config)

# Keep trainable adapters in FP32; frozen base weights remain FP16.
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        assert "lora_" in name, f"Unexpected trainable parameter: {name}"
        parameter.data = parameter.data.float()
    else:
        assert parameter.dtype == torch.float16, (name, parameter.dtype)
del parameter

trainable = [
    (name, parameter)
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
]
assert trainable
assert all(p.dtype == torch.float32 for _, p in trainable)
assert all(p.device == DEVICE for _, p in trainable)

b_parameters = [p for name, p in trainable if "lora_B" in name]
assert b_parameters
assert all(torch.count_nonzero(p).item() == 0 for p in b_parameters)

model.print_trainable_parameters()
model.save_pretrained(
    str(INITIAL_ADAPTER_DIR),
    safe_serialization=True,
)

adapter_file = INITIAL_ADAPTER_DIR / "adapter_model.safetensors"
assert adapter_file.is_file()
assert (INITIAL_ADAPTER_DIR / "adapter_config.json").is_file()

initialization = {
    "model": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "initialization_seed": SEED,
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": "all-linear",
    "base_dtype": "float16",
    "adapter_dtype": "float32",
    "attention_implementation": "sdpa",
    "trainable_parameters": sum(p.numel() for _, p in trainable),
    "total_parameters": sum(p.numel() for p in model.parameters()),
    "initial_adapter_sha256": hashlib.sha256(
        adapter_file.read_bytes()
    ).hexdigest(),
    "optimizer_updates": 0,
}
INITIALIZATION_PATH.write_text(
    json.dumps(initialization, indent=2), encoding="utf-8"
)

del trainable, b_parameters
print(json.dumps(initialization, indent=2))
print("Saved:", INITIAL_ADAPTER_DIR)
print("No training or validation/test evaluation performed.")

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338
{
  "model": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
  "model_revision": "fe8a4ea1ffedaf415f4da2f062534de366a451e6",
  "initialization_seed": 42,
  "lora_rank": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "target_modules": "all-linear",
  "base_dtype": "float16",
  "adapter_dtype": "float32",
  "attention_implementation": "sdpa",
  "trainable_parameters": 12615680,
  "total_parameters": 1112664064,
  "initial_adapter_sha256": "9c237fcff8f91caa905be99d12cf6ae1898a9d56000e50f86767784f0f095ec3",
  "optimizer_updates": 0
}
Saved: /kaggle/working/HealthVer_TinyLlama_Experiments/initial_adapter_seed42
No training or validation/test evaluation performed.


In [6]:
import gc
import json
import math
import torch

def forward_backward_check():
    batch_size = 8
    scale = 1024.0
    indices = sorted(
        range(len(full_train)),
        key=lambda i: full_train[i]["sequence_length"],
        reverse=True,
    )[:batch_size]
    rows = [full_train[i] for i in indices]
    padded_length = max(len(row["input_ids"]) for row in rows)
    assert padded_length <= SEQUENCE_LIMIT

    batch = {}
    for key, padding_value in [
        ("input_ids", tokenizer.pad_token_id),
        ("attention_mask", 0),
        ("labels", -100),
    ]:
        batch[key] = torch.tensor(
            [
                row[key] + [padding_value] * (
                    padded_length - len(row[key])
                )
                for row in rows
            ],
            dtype=torch.long,
            device=DEVICE,
        )

    # Mask padding by position; EOS and PAD share the same token ID.
    assert torch.all(
        batch["labels"][batch["attention_mask"] == 0] == -100
    )
    previous_training_mode = model.training
    model.zero_grad(set_to_none=True)
    model.train()
    torch.cuda.synchronize(DEVICE)
    torch.cuda.reset_peak_memory_stats(DEVICE)

    try:
        with torch.random.fork_rng(devices=[DEVICE.index]):
            torch.random.default_generator.manual_seed(SEED)
            with torch.cuda.device(DEVICE):
                torch.cuda.manual_seed(SEED)

            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(**batch, use_cache=False).loss

            assert torch.isfinite(loss).item(), "Non-finite loss."
            (loss * scale).backward()

        squared_norm = 0.0
        gradient_tensors = 0
        for name, parameter in model.named_parameters():
            if not parameter.requires_grad:
                continue
            assert parameter.grad is not None, f"Missing gradient: {name}"
            gradient = parameter.grad.detach().float() / scale
            assert torch.isfinite(gradient).all().item(), (
                f"Non-finite gradient: {name}"
            )
            squared_norm += gradient.double().square().sum().item()
            gradient_tensors += 1

        assert math.isfinite(squared_norm) and squared_norm > 0
        torch.cuda.synchronize(DEVICE)
        return {
            "batch_size": batch_size,
            "padded_sequence_length": padded_length,
            "loss": loss.item(),
            "unscaled_adapter_gradient_norm": math.sqrt(squared_norm),
            "adapter_tensors_with_gradients": gradient_tensors,
            "peak_allocated_gib":
                torch.cuda.max_memory_allocated(DEVICE) / 1024**3,
            "optimizer_updates": 0,
        }
    finally:
        model.zero_grad(set_to_none=True)
        model.train(previous_training_mode)

memory_report = forward_backward_check()
gc.collect()
torch.cuda.empty_cache()

print(json.dumps(memory_report, indent=2))
print("Forward/backward check passed; no optimizer update performed.")
print("No files written or validation/test evaluation performed.")

{
  "batch_size": 8,
  "padded_sequence_length": 409,
  "loss": 8.226152420043945,
  "unscaled_adapter_gradient_norm": 52.59194643173037,
  "adapter_tensors_with_gradients": 308,
  "peak_allocated_gib": 11.505407333374023,
  "optimizer_updates": 0
}
Forward/backward check passed; no optimizer update performed.
No files written or validation/test evaluation performed.


In [7]:
import gc
import json
import math
import time
import hashlib
from pathlib import Path
from importlib.metadata import version

import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, GenerationConfig,
    Trainer, TrainingArguments, set_seed,
)
from transformers.trainer_utils import get_last_checkpoint
from peft import PeftModel
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
)
from tqdm.auto import tqdm

from datasets import load_from_disk



def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024**2), b''):
            h.update(chunk)
    return h.hexdigest()

def json_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)

def content_hash(dataset):
    h = hashlib.sha256()
    for row in dataset:
        h.update((json.dumps(row, sort_keys=True, separators=(',', ':')) + '\n').encode('utf-8'))
    return h.hexdigest()


EXPERIMENT_ROOT = Path('/kaggle/working/HealthVer_TinyLlama_Experiments')
BASELINE_ROOT = Path('/kaggle/input/datasets/muusamunufo/base-1-trained/HealthVer_Experiments')
RUN_DIR = EXPERIMENT_ROOT / 'TinyLlama_LoRA_full_seed42'
INITIAL_ADAPTER = EXPERIMENT_ROOT / 'initial_adapter_seed42'
DEVICE = torch.device('cuda:0')
assert torch.cuda.is_available()
torch.cuda.set_device(DEVICE)

input_config = json.loads((EXPERIMENT_ROOT / 'input_config.json').read_text())
preprocessing = json.loads((EXPERIMENT_ROOT / 'preprocessing_manifest.json').read_text())
initialization = json.loads((EXPERIMENT_ROOT / 'initialization_manifest.json').read_text())
dataset_metadata = json.loads((BASELINE_ROOT / 'dataset_metadata.json').read_text())
MODEL_ID = input_config['model']
MODEL_REVISION = input_config['model_revision']
DATASET_REVISION = input_config['dataset_revision']
LABELS = input_config['labels']
SEQUENCE_LIMIT = input_config['max_sequence_length']
assert MODEL_ID == 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
assert MODEL_REVISION == 'fe8a4ea1ffedaf415f4da2f062534de366a451e6'
assert DATASET_REVISION == dataset_metadata['revision'] == 'b20ac99ceed62f5264a319fa25a854df1668d85b'
assert LABELS == ['Supports', 'Refutes', 'Neutral']
assert SEQUENCE_LIMIT == preprocessing['sequence_limit'] == 448
assert input_config['truncation'] is False
assert input_config['loss'] == 'assistant_answer_only'
assert initialization['optimizer_updates'] == 0
assert initialization['trainable_parameters'] == 12615680
assert sha256(INITIAL_ADAPTER / 'adapter_model.safetensors') == initialization['initial_adapter_sha256']
assert initialization['initial_adapter_sha256'] == '9c237fcff8f91caa905be99d12cf6ae1898a9d56000e50f86767784f0f095ec3'

package_versions = {name: version(name) for name in [
    'torch', 'transformers', 'peft', 'accelerate', 'datasets', 'scikit-learn',
]}
assert package_versions == {
    'torch': '2.11.0+cu128', 'transformers': '5.16.1', 'peft': '0.20.0',
    'accelerate': '1.14.0', 'datasets': '4.8.5', 'scikit-learn': '1.6.1',
}, package_versions

tokenizer = AutoTokenizer.from_pretrained(str(EXPERIMENT_ROOT / 'tokenizer'), local_files_only=True)
tokenizer.padding_side = 'right'
assert tokenizer.pad_token_id is not None and tokenizer.eos_token_id is not None
generation_config = GenerationConfig(
    max_new_tokens=12, do_sample=False, num_beams=1, use_cache=True,
    pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
    bos_token_id=tokenizer.bos_token_id,
)
SEED, EPOCHS, MICRO_BATCH, GRAD_ACCUM = 42, 3, 8, 4


def load_original(split, filename):
    path = BASELINE_ROOT / 'original_data' / f'{DATASET_REVISION}_{filename}'
    assert sha256(path) == dataset_metadata['files'][split]['sha256']
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert frame['id'].is_unique and set(frame['label']).issubset(LABELS)
    return frame


train_frame = load_original('train', 'healthver_train.csv')
validation_frame = load_original('validation', 'healthver_dev.csv')
full_train = load_from_disk(str(EXPERIMENT_ROOT / 'tokenized_train'))
full_validation = load_from_disk(str(EXPERIMENT_ROOT / 'tokenized_validation'))
assert len(full_train) == len(train_frame) == 10590
assert len(full_validation) == len(validation_frame) == 1917
for split, data, frame in [
    ('train', full_train, train_frame), ('validation', full_validation, validation_frame),
]:
    manifest = preprocessing['splits'][split]
    assert list(data['id']) == frame['id'].tolist()
    assert content_hash(data) == manifest['tokenized_content_sha256']
    state = json.loads((EXPERIMENT_ROOT / f'tokenized_{split}' / 'state.json').read_text())
    assert state['_fingerprint'] == manifest['dataset_fingerprint']
    assert max(data['sequence_length']) <= SEQUENCE_LIMIT

label_lookup = {label.casefold(): label for label in LABELS}


def make_messages(claim, evidence):
    return [
        {'role': 'system', 'content': input_config['system_prompt']},
        {'role': 'user', 'content': f'Claim: {claim}\n\nEvidence: {evidence}'},
    ]

def parse_prediction(response):
    return label_lookup.get(' '.join(response.strip().split()).casefold(), 'INVALID')

def training_collator(rows):
    padding = {'input_ids': tokenizer.pad_token_id, 'attention_mask': 0, 'labels': -100}
    batch = {
        key: pad_sequence([torch.tensor(row[key], dtype=torch.long) for row in rows],
                          batch_first=True, padding_value=value)
        for key, value in padding.items()
    }
    assert batch['input_ids'].shape[1] <= SEQUENCE_LIMIT
    return batch

def evaluate_labels(eval_model, frame, folder, split, fingerprint):
    folder.mkdir(parents=True, exist_ok=True)
    previous_padding, previous_mode = tokenizer.padding_side, eval_model.training
    records = []
    started = time.perf_counter()
    max_prompt_tokens = 0
    try:
        tokenizer.padding_side = 'left'
        eval_model.eval()
        with torch.inference_mode():
            for start in tqdm(range(0, len(frame), 8), desc=f'{folder.parent.name}: {split}', leave=False):
                rows = frame.iloc[start:start + 8]
                prompts = [tokenizer.apply_chat_template(
                    make_messages(row.claim, row.evidence), tokenize=False, add_generation_prompt=True,
                ) for row in rows.itertuples(index=False)]
                inputs = tokenizer(prompts, padding=True, truncation=False,
                                   add_special_tokens=False, return_tensors='pt').to(DEVICE)
                width = inputs['input_ids'].shape[1]
                max_prompt_tokens = max(max_prompt_tokens, width)
                limit = SEQUENCE_LIMIT if split == 'validation' else eval_model.config.max_position_embeddings
                assert width + generation_config.max_new_tokens <= limit
                generated = eval_model.generate(**inputs, generation_config=generation_config)
                responses = tokenizer.batch_decode(generated[:, width:], skip_special_tokens=True)
                for uid, gold, response in zip(rows['id'], rows['label'], responses):
                    records.append({'id': uid, 'true_label': gold,
                                    'predicted_label': parse_prediction(response), 'raw_response': response})
                del inputs, generated
        assert [r['id'] for r in records] == frame['id'].tolist()
        gold, predicted = [r['true_label'] for r in records], [r['predicted_label'] for r in records]
        metrics = {
            'evaluation_split': split, 'n_examples': len(records), 'run_config_sha256': fingerprint,
            'source_csv_sha256': dataset_metadata['files'][split]['sha256'],
            'accuracy': float(accuracy_score(gold, predicted)),
            'macro_f1': float(f1_score(gold, predicted, labels=LABELS, average='macro', zero_division=0)),
            'weighted_f1': float(f1_score(gold, predicted, labels=LABELS, average='weighted', zero_division=0)),
            'invalid_rate': sum(p == 'INVALID' for p in predicted) / len(gold),
            'max_prompt_tokens': max_prompt_tokens, 'truncation': False,
            'elapsed_seconds': time.perf_counter() - started,
        }
        with (folder / 'predictions.jsonl').open('w', encoding='utf-8') as handle:
            for record in records:
                handle.write(json.dumps(record) + '\n')
        save_json(folder / 'classification_report.json', classification_report(
            gold, predicted, labels=LABELS, output_dict=True, zero_division=0))
        matrix_labels = LABELS + ['INVALID']
        pd.DataFrame(confusion_matrix(gold, predicted, labels=matrix_labels),
                     index=matrix_labels, columns=matrix_labels).loc[LABELS].to_csv(folder / 'confusion_matrix.csv')
        save_json(folder / 'metrics.json', metrics)
        return metrics
    finally:
        tokenizer.padding_side = previous_padding
        eval_model.train(previous_mode)
        gc.collect()
        torch.cuda.empty_cache()

class SingleGPUArguments(TrainingArguments):
    @property
    def n_gpu(self):
        return 1

class FullDataTrainer(Trainer):
    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix='eval', **kwargs):
        result = evaluate_labels(self.model, validation_frame,
                                 Path(self.args.output_dir) / 'validation' / f'step_{self.state.global_step}',
                                 'validation', self.run_fingerprint)
        metrics = {f'{metric_key_prefix}_{key}': result[key]
                   for key in ['accuracy', 'macro_f1', 'weighted_f1', 'invalid_rate']}
        metrics[f'{metric_key_prefix}_runtime'] = result['elapsed_seconds']
        self.log(metrics)
        self.control = self.callback_handler.on_evaluate(self.args, self.state, self.control, metrics)
        return metrics


def train_full_data():
    expected_steps = EPOCHS * math.ceil(math.ceil(len(full_train) / MICRO_BATCH) / GRAD_ACCUM)
    warmup_steps = math.ceil(0.03 * expected_steps)
    assert (expected_steps, warmup_steps) == (993, 30)
    run_config = {
        'model': MODEL_ID, 'model_revision': MODEL_REVISION, 'method': 'full_data',
        'dataset_revision': DATASET_REVISION, 'training_rows': len(full_train),
        'validation_rows': len(full_validation), 'seed': SEED, 'epochs': EPOCHS,
        'micro_batch_size': MICRO_BATCH, 'gradient_accumulation': GRAD_ACCUM,
        'learning_rate': 1e-4, 'optimizer': 'adamw_torch', 'weight_decay': 0.01,
        'max_grad_norm': 1.0, 'scheduler': 'linear', 'warmup_steps': warmup_steps,
        'expected_optimizer_steps': expected_steps, 'sequence_limit': SEQUENCE_LIMIT,
        'truncation': False, 'loss': 'assistant_answer_only', 'padding': 'dynamic',
        'selection_metric': 'validation_macro_f1', 'evaluation_batch_size': 8,
        'fp16': True, 'bf16': False, 'gradient_checkpointing': False,
        'base_dtype': 'float16', 'adapter_dtype': 'float32', 'attention_implementation': 'sdpa',
        'lora_rank': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'target_modules': 'all-linear',
        'initial_adapter_sha256': initialization['initial_adapter_sha256'],
        'initial_adapter_config_sha256': sha256(INITIAL_ADAPTER / 'adapter_config.json'),
        'input_config_sha256': sha256(EXPERIMENT_ROOT / 'input_config.json'),
        'preprocessing_manifest_sha256': sha256(EXPERIMENT_ROOT / 'preprocessing_manifest.json'),
        'chat_template_sha256': hashlib.sha256(tokenizer.chat_template.encode()).hexdigest(),
        'training_fingerprint': preprocessing['splits']['train']['dataset_fingerprint'],
        'validation_fingerprint': preprocessing['splits']['validation']['dataset_fingerprint'],
        'training_content_sha256': preprocessing['splits']['train']['tokenized_content_sha256'],
        'validation_content_sha256': preprocessing['splits']['validation']['tokenized_content_sha256'],
        'fingerprint_source': 'persisted_dataset_state',
        'generation_config': generation_config.to_dict(), 'package_versions': package_versions,
    }
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    config_path = RUN_DIR / 'run_config.json'
    if config_path.exists():
        assert json.loads(config_path.read_text()) == run_config, 'Existing run configuration differs.'
    else:
        assert not list(RUN_DIR.glob('checkpoint-*')), 'Checkpoints exist without run configuration.'
        save_json(config_path, run_config)
    fingerprint = json_hash(run_config)
    completion_path = RUN_DIR / 'training_complete.json'
    if completion_path.exists():
        complete = json.loads(completion_path.read_text())
        assert complete['run_config_sha256'] == fingerprint
        assert sha256(RUN_DIR / 'best_adapter/adapter_model.safetensors') == complete['best_adapter_sha256']
        print('Verified completed full-data training; no retraining performed.')
        return complete

    checkpoint = get_last_checkpoint(str(RUN_DIR))
    if checkpoint is not None:
        for filename in ['adapter_model.safetensors', 'adapter_config.json', 'optimizer.pt',
                         'scheduler.pt', 'trainer_state.json', 'rng_state.pth', 'scaler.pt']:
            assert (Path(checkpoint) / filename).is_file(), f'Incomplete checkpoint: {filename}'

    local_base = local_model = local_trainer = None
    try:
        set_seed(SEED)
        tokenizer.padding_side = 'right'
        local_base = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, revision=MODEL_REVISION, dtype=torch.float16,
            attn_implementation='sdpa', device_map={'': 'cuda:0'},
        )
        local_model = PeftModel.from_pretrained(local_base, str(INITIAL_ADAPTER), is_trainable=True)
        local_model.config.use_cache = False
        local_model.config.pad_token_id = tokenizer.pad_token_id
        local_model.generation_config = generation_config
        local_model.gradient_checkpointing_disable()
        for parameter_name, parameter in local_model.named_parameters():
            if parameter.requires_grad:
                assert 'lora_' in parameter_name
                parameter.data = parameter.data.float()
        del parameter
        assert sum(p.numel() for p in local_model.parameters() if p.requires_grad) == 12615680

        args = SingleGPUArguments(
            output_dir=str(RUN_DIR), num_train_epochs=EPOCHS,
            per_device_train_batch_size=MICRO_BATCH, gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=1e-4, warmup_steps=warmup_steps,
            lr_scheduler_type='linear', optim='adamw_torch', weight_decay=0.01, max_grad_norm=1.0,
            fp16=True, bf16=False, gradient_checkpointing=False,
            eval_strategy='epoch', save_strategy='epoch', save_total_limit=2, save_only_model=False,
            load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
            logging_steps=25, logging_first_step=True, report_to='none',
            dataloader_num_workers=0, dataloader_pin_memory=True,
            seed=SEED, data_seed=SEED, label_names=['labels'],
        )
        assert args.world_size == 1 and args.device == DEVICE and args.train_batch_size == MICRO_BATCH
        (RUN_DIR / 'training_args.json').write_text(args.to_json_string(), encoding='utf-8')
        local_trainer = FullDataTrainer(
            model=local_model, args=args, train_dataset=full_train, eval_dataset=full_validation,
            data_collator=training_collator, processing_class=tokenizer,
        )
        local_trainer.run_fingerprint = fingerprint
        print('Starting from:', checkpoint or INITIAL_ADAPTER)
        print('Expected optimizer steps:', expected_steps, '| Warmup steps:', warmup_steps)
        torch.cuda.reset_peak_memory_stats(DEVICE)
        result = local_trainer.train(resume_from_checkpoint=checkpoint)
        assert local_trainer.state.global_step == expected_steps
        assert local_trainer.state.epoch >= EPOCHS - 1e-6
        assert local_trainer.state.best_metric is not None
        validation_scores = [entry['eval_macro_f1'] for entry in local_trainer.state.log_history
                             if 'eval_macro_f1' in entry]
        assert len(validation_scores) == EPOCHS
        assert abs(max(validation_scores) - local_trainer.state.best_metric) < 1e-12
        local_trainer.save_model(str(RUN_DIR / 'best_adapter'))
        tokenizer.save_pretrained(str(RUN_DIR / 'best_adapter'))
        local_trainer.save_metrics('train', result.metrics)
        local_trainer.save_state()
        complete = {
            'run_config_sha256': fingerprint,
            'best_validation_macro_f1': float(local_trainer.state.best_metric),
            'best_checkpoint': local_trainer.state.best_model_checkpoint,
            'global_step': int(local_trainer.state.global_step), 'epoch': float(local_trainer.state.epoch),
            'train_call_seconds': result.metrics['train_runtime'],
            'peak_allocated_gib': torch.cuda.max_memory_allocated(DEVICE) / 1024**3,
            'best_adapter_sha256': sha256(RUN_DIR / 'best_adapter/adapter_model.safetensors'),
            'resumed_from': checkpoint, 'runtime_note': 'A resumed call excludes earlier sessions.',
        }
        assert sha256(INITIAL_ADAPTER / 'adapter_model.safetensors') == initialization['initial_adapter_sha256']
        save_json(completion_path, complete)
        return complete
    finally:
        local_trainer = local_model = local_base = None
        gc.collect()
        torch.cuda.empty_cache()

# Release the model used in the memory check before loading fresh weights.
for old_name in ['trainer', 'model']:
    if old_name in globals():
        del globals()[old_name]
gc.collect()
torch.cuda.empty_cache()

training_completion = train_full_data()
print(json.dumps(training_completion, indent=2))
print('Saved:', RUN_DIR / 'best_adapter')
print('Full-data training complete. Test data not accessed.')


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Starting from: /kaggle/working/HealthVer_TinyLlama_Experiments/initial_adapter_seed42
Expected optimizer steps: 993 | Warmup steps: 30


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Invalid Rate
1,0.121227,No log,0.775170,0.728231,0.768849,0.000000
2,0.069626,No log,0.793949,0.767025,0.798835,0.000000
3,0.027135,No log,0.818988,0.790851,0.819798,0.000000


validation: validation:   0%|          | 0/240 [00:00<?, ?it/s]

validation: validation:   0%|          | 0/240 [00:00<?, ?it/s]

validation: validation:   0%|          | 0/240 [00:00<?, ?it/s]

{
  "run_config_sha256": "55ba86f1d6186f893f117a9700b3e362c8ee07892d1a12f653afea18c2e4eec9",
  "best_validation_macro_f1": 0.7908511534803102,
  "best_checkpoint": "/kaggle/working/HealthVer_TinyLlama_Experiments/TinyLlama_LoRA_full_seed42/checkpoint-993",
  "global_step": 993,
  "epoch": 3.0,
  "train_call_seconds": 4035.2966,
  "peak_allocated_gib": 11.68085527420044,
  "best_adapter_sha256": "2fa77dfd4a3b264b40daabebc3d98e3e37465e78c0f5214bd290553a2c0729b1",
  "resumed_from": null,
  "runtime_note": "A resumed call excludes earlier sessions."
}
Saved: /kaggle/working/HealthVer_TinyLlama_Experiments/TinyLlama_LoRA_full_seed42/best_adapter
Full-data training complete. Test data not accessed.


In [8]:
import gc
import json
import shutil
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, GenerationConfig
from peft import PeftModel

run_config = json.loads((RUN_DIR / "run_config.json").read_text())
completion = json.loads((RUN_DIR / "training_complete.json").read_text())
run_fingerprint = json_hash(run_config)
BEST_ADAPTER = RUN_DIR / "best_adapter"
TEST_DIR = RUN_DIR / "test"

assert completion["run_config_sha256"] == run_fingerprint
assert completion["global_step"] == 993
assert completion["epoch"] == 3.0
assert sha256(BEST_ADAPTER / "adapter_model.safetensors") == (
    completion["best_adapter_sha256"]
)

# Use the generation settings frozen before training.
generation_config = GenerationConfig.from_dict(
    run_config["generation_config"]
)
assert generation_config.max_new_tokens == 12
assert generation_config.do_sample is False
assert generation_config.num_beams == 1

# First test-data access: after the best adapter has been fixed.
test_frame = load_original("test", "healthver_test.csv")
assert len(test_frame) == 1823
assert set(test_frame["id"]).isdisjoint(train_frame["id"])
assert set(test_frame["id"]).isdisjoint(validation_frame["id"])

# Preserve the original test CSV for future baseline reuse.
source_test = (
    BASELINE_ROOT / "original_data"
    / f"{DATASET_REVISION}_healthver_test.csv"
)
saved_test = EXPERIMENT_ROOT / "original_data" / source_test.name
if saved_test.exists():
    assert sha256(saved_test) == dataset_metadata["files"]["test"]["sha256"]
else:
    shutil.copy2(source_test, saved_test)
    assert sha256(saved_test) == dataset_metadata["files"]["test"]["sha256"]

metrics_path = TEST_DIR / "metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    assert metrics["run_config_sha256"] == run_fingerprint
    assert metrics["source_csv_sha256"] == (
        dataset_metadata["files"]["test"]["sha256"]
    )
    assert metrics["n_examples"] == len(test_frame)
    assert (TEST_DIR / "predictions.jsonl").is_file()
    print("Reusing saved test metrics for this run.")
else:
    test_base = test_model = None
    try:
        gc.collect()
        torch.cuda.empty_cache()

        test_base = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            revision=MODEL_REVISION,
            dtype=torch.float16,
            attn_implementation="sdpa",
            device_map={"": "cuda:0"},
        )
        test_model = PeftModel.from_pretrained(
            test_base, str(BEST_ADAPTER), is_trainable=False
        )
        test_model.config.pad_token_id = tokenizer.pad_token_id
        test_model.generation_config = generation_config
        test_model.eval()
        assert not any(p.requires_grad for p in test_model.parameters())

        metrics = evaluate_labels(
            test_model, test_frame, TEST_DIR, "test", run_fingerprint
        )
    finally:
        test_model = test_base = None
        gc.collect()
        torch.cuda.empty_cache()

summary = pd.DataFrame([{
    "model": MODEL_ID,
    "method": "full_data",
    "training_rows": run_config["training_rows"],
    "training_seed": run_config["seed"],
    "best_validation_macro_f1": completion["best_validation_macro_f1"],
    "test_accuracy": metrics["accuracy"],
    "test_macro_f1": metrics["macro_f1"],
    "test_weighted_f1": metrics["weighted_f1"],
    "test_invalid_rate": metrics["invalid_rate"],
    "train_call_seconds": completion["train_call_seconds"],
    "test_seconds": metrics["elapsed_seconds"],
}])
summary.to_csv(EXPERIMENT_ROOT / "full_data_results.csv", index=False)

print(summary.to_string(index=False))
print("Predictions and evaluation reports:", TEST_DIR)
print("Results:", EXPERIMENT_ROOT / "full_data_results.csv")
print("Full-data baseline complete. No optimizer updates performed.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

TinyLlama_LoRA_full_seed42: test:   0%|          | 0/228 [00:00<?, ?it/s]

                             model    method  training_rows  training_seed  best_validation_macro_f1  test_accuracy  test_macro_f1  test_weighted_f1  test_invalid_rate  train_call_seconds  test_seconds
TinyLlama/TinyLlama-1.1B-Chat-v1.0 full_data          10590             42                  0.790851       0.805266       0.796657          0.804839                0.0           4035.2966    131.476584
Predictions and evaluation reports: /kaggle/working/HealthVer_TinyLlama_Experiments/TinyLlama_LoRA_full_seed42/test
Results: /kaggle/working/HealthVer_TinyLlama_Experiments/full_data_results.csv
Full-data baseline complete. No optimizer updates performed.
